## 1. Setup e download dos dados

In [22]:
%pip install -q huggingface_hub hf_xet

Note: you may need to restart the kernel to use updated packages.


In [23]:
import json
import re
import copy
import random
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter

from huggingface_hub import hf_hub_download

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Padrão de tag de orador nas transcrições da Câmara: "(Nome. Partido - UF)".
SPEAKER_TAG_PATTERN = re.compile(r"\(([^()]{3,80})\)")

REPO_ID = "unicamp-dl/PublicHearingBR"
FILES = {
    "lds": "PublicHearingBR_LDS.jsonl",
    "nli": "PublicHearingBR_NLI.jsonl",
}


def download_file(filename: str) -> Path:
    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=filename,
        local_dir=DATA_DIR,
    )
    print(f"[ok] {filename} -> {path}")
    return Path(path)

lds_path = download_file(FILES["lds"])
nli_path = download_file(FILES["nli"])

[ok] PublicHearingBR_LDS.jsonl -> /home/lmmf/Documentos/kunumi/ideias_em_rede/OpenParla/Leo/data/PublicHearingBR_LDS.jsonl
[ok] PublicHearingBR_NLI.jsonl -> /home/lmmf/Documentos/kunumi/ideias_em_rede/OpenParla/Leo/data/PublicHearingBR_NLI.jsonl


In [24]:
def load_jsonl(path: Path):
    registros = []
    with open(path, encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                registros.append(json.loads(linha))
    return registros

phbr_lds = load_jsonl(lds_path)
phbr_nli = load_jsonl(nli_path)

print(f"PublicHearingBR_LDS: {len(phbr_lds)} registros")
print(f"PublicHearingBR_NLI: {len(phbr_nli)} registros")

PublicHearingBR_LDS: 206 registros
PublicHearingBR_NLI: 206 registros


## 2. Diagnóstico: rode isto antes de confiar no resto

Observar amostra real de 'cargo' e de tags de orador extraídas de 'transcricao'

In [25]:
def coletar_cargos(dataset, chave_envolvidos):
    cargos = []
    node_key, sub_key = chave_envolvidos
    for r in dataset:
        for env in r.get(node_key, {}).get(sub_key, []):
            cargos.append(env.get("cargo", ""))
    return cargos

cargos_lds = coletar_cargos(phbr_lds, ("metadados", "envolvidos"))
cargos_nli = coletar_cargos(phbr_nli, ("metadados_extraidos", "envolvidos"))

print(f"Total de envolvidos (LDS): {len(cargos_lds)}")
print(f"Total de envolvidos (NLI): {len(cargos_nli)}")

print("\nAmostra de 'cargo' no LDS:")
for c in random.sample(cargos_lds, min(20, len(cargos_lds))):
    print(" -", c)

print("\nAmostra de 'cargo' no NLI:")
for c in random.sample(cargos_nli, min(20, len(cargos_nli))):
    print(" -", c)

Total de envolvidos (LDS): 1065
Total de envolvidos (NLI): 1413

Amostra de 'cargo' no LDS:
 - Diretor de políticas públicas da Uber
 - Magistrado do Tribunal de Justiça do Paraná
 - Deputado (Psol-SP)
 - Deputado (PSB-SP)
 - Presidente do Apoio à Pesquisa e Pacientes de Cannabis Medicinal do Rio de Janeiro
 - Deputado (PT-PI)
 - Vice-presidente da Associação Nacional dos Pós-Graduandos (ANPG)
 - Deputado (PL-AM)
 - Deputado (PSD-RJ)
 - Diretor de política agrícola da Companhia Nacional de Abastecimento (Conab)
 - Promotor do Ministério Público do Distrito Federal e Territórios (MPDFT)
 - Coordenadora de campanhas da Proteção Animal Mundial (WSPA)
 - Deputado (PT-SP)
 - Integrante da Articulação Social de Luta contra a Tuberculose
 - Mestre em Direito e ex-presidente nacional da Ordem dos Advogados do Brasil (OAB)
 - Neurologista
 - Diretor de Gestão de Fundos do Ministério do Trabalho e Emprego
 - Defensor público da União
 - Presidente do Instituto Distonia Saúde
 - Deputada (Psol-SP

In [26]:
amostra_tags = []
for r in random.sample(phbr_lds, min(5, len(phbr_lds))):
    tags = SPEAKER_TAG_PATTERN.findall(r.get("transcricao", ""))
    amostra_tags.extend(tags[:10])

print("Amostra de conteúdo entre parênteses na transcrição (procurando o padrão 'Nome. Partido - UF'):")
for t in amostra_tags[:40]:
    print(" -", t)

Amostra de conteúdo entre parênteses na transcrição (procurando o padrão 'Nome. Partido - UF'):
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Prof. Paulo Fernando. Bloco/REPUBLICANOS - DF
 - Paulo Litro. Bloco/PSD - PR
 - Bibo Nunes. PL - RS
 - Palmas.
 - risos
 - Bibo Nunes. PL - RS
 - Bibo Nunes. PL - RS
 - Bibo Nunes. PL - RS
 - Bibo Nunes. PL - RS
 - Bibo Nunes. PL - RS
 - Bibo Nunes. PL - RS
 - Paulão. Bloco/PT - AL
 - Paulão. Bloco/PT - AL
 - Paulão. Bloco/PT - AL
 - Paulão. Bloco/PT - AL
 - Paulão. Bloco/PT - AL
 - Paulão. Bloco/PT - AL
 - Pausa.
 - Paulão. Bloco/PT - AL
 - Gisela Simona.

## 3. Remoção de jornalistas

In [27]:
def normalize(texto: str) -> str:
    if not texto:
        return ""
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()

JOURNALIST_KEYWORDS = [
    "jornalista", "correspondente", "reporter", "apresentador", "apresentadora",
    "colunista", "comentarista", "blogueiro", "blogueira", "redator", "redatora",
    "ancora", "editor-chefe", "editora-chefe", "editor de jornal", "editora de jornal",
    "imprensa",
]

def is_journalist(cargo: str) -> bool:
    cargo_norm = normalize(cargo)
    return any(kw in cargo_norm for kw in JOURNALIST_KEYWORDS)


## 4. Extração do estado (UF)

In [28]:
UF_LIST = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS",
    "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC",
    "SP", "SE", "TO",
]
UF_SET = set(UF_LIST)

# Ex.: "PSDB - RS", "Bloco/UNIÃO - MG", ".../SP"
CARGO_UF_PATTERN = re.compile(r"(?:-\s*|/)\s*([A-Z]{2})\b")

# Fallback pra quando o cargo escreve o estado por extenso em vez de sigla
# (ex.: "Deputado Estadual pelo PSOL de São Paulo"). Mantém acento e não usa
# normalize() de propósito: sem acento "Pará" vira "para", a preposição mais
# comum do português, e isso geraria falso positivo em qualquer cargo.
UF_NOME_COMPLETO = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF",
    "Espírito Santo": "ES", "Goiás": "GO", "Maranhão": "MA",
    "Mato Grosso do Sul": "MS", "Mato Grosso": "MT", "Minas Gerais": "MG",
    "Pará": "PA", "Paraíba": "PB", "Paraná": "PR", "Pernambuco": "PE",
    "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR",
    "Santa Catarina": "SC", "São Paulo": "SP", "Sergipe": "SE",
    "Tocantins": "TO",
}
# nomes mais longos primeiro
_NOMES_UF_ORDENADOS = sorted(UF_NOME_COMPLETO, key=len, reverse=True)
NOME_UF_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(n) for n in _NOMES_UF_ORDENADOS) + r")\b",
    re.IGNORECASE,
)

def extract_uf_from_cargo(cargo: str):
    if not cargo:
        return None
    for m in CARGO_UF_PATTERN.finditer(cargo):
        if m.group(1) in UF_SET:
            return m.group(1)
    m2 = NOME_UF_PATTERN.search(cargo)
    if m2:
        chave = next(k for k in UF_NOME_COMPLETO if k.lower() == m2.group(1).lower())
        return UF_NOME_COMPLETO[chave]
    return None

NAME_BEFORE_PAREN_PATTERN = re.compile(
    r"([A-ZÀ-ÜÇ][A-ZÀ-ÜÇ\.\s]{2,60}?)\s*\(([^()]{1,40}-\s*[A-Z]{2})\)"
)

def build_speaker_uf_map(transcricao: str) -> dict:
    """Varre a transcrição em busca de tags de orador e devolve
    {nome_normalizado: uf}. Tenta duas formas (ver comentário acima)."""
    mapping = {}
    if not transcricao:
        return mapping

    #"(Nome. Partido - UF)" tudo dentro do parênteses
    for content in SPEAKER_TAG_PATTERN.findall(transcricao):
        if "." not in content:
            continue
        nome_part, _, resto = content.partition(".")
        nome = nome_part.strip()
        m = re.search(r"-\s*([A-Z]{2})\s*$", resto.strip())
        if nome and m and m.group(1) in UF_SET:
            mapping[normalize(nome)] = m.group(1)

    #fallback: nome antes do parênteses, só "Partido - UF" dentro
    for m in NAME_BEFORE_PAREN_PATTERN.finditer(transcricao):
        nome, resto = m.group(1).strip(" ."), m.group(2)
        if "." in resto:
            continue  # já coberto pela forma acima; evita capturar "PRESIDENTE"/"O SR" como nome
        m_uf = re.search(r"-\s*([A-Z]{2})\s*$", resto.strip())
        if nome and m_uf and m_uf.group(1) in UF_SET:
            mapping.setdefault(normalize(nome), m_uf.group(1))

    return mapping


def match_name_to_uf(nome: str, speaker_map: dict):
    nome_norm = normalize(nome)
    if not nome_norm or not speaker_map:
        return None
    if nome_norm in speaker_map:
        return speaker_map[nome_norm]
    nome_tokens = set(nome_norm.split())
    for speaker_name, uf in speaker_map.items():
        if nome_norm in speaker_name or speaker_name in nome_norm:
            return uf
        # exige >=2 tokens em comum pra evitar falso-positivo por sobrenome comum
        if len(nome_tokens & set(speaker_name.split())) >= 2:
            return uf
    return None

def resolve_uf(envolvido: dict, speaker_map: dict):
    uf = extract_uf_from_cargo(envolvido.get("cargo", ""))
    if uf:
        return uf
    return match_name_to_uf(envolvido.get("nome", ""), speaker_map)

## 5. Separar por estado e salvar

In [29]:
def process_record(record: dict, envolvidos_path: tuple, usa_transcricao: bool):
    """Devolve (registro_com_envolvidos_filtrados, ufs_encontradas, n_jornalistas_removidos).
    A estrutura original do registro é preservada — só a lista de envolvidos é filtrada."""
    novo = copy.deepcopy(record)
    node_key, sub_key = envolvidos_path
    envolvidos = novo.get(node_key, {}).get(sub_key, [])

    speaker_map = build_speaker_uf_map(record.get("transcricao", "")) if usa_transcricao else {}

    mantidos, ufs_encontradas, n_jornalistas = [], set(), 0
    for env in envolvidos:
        if is_journalist(env.get("cargo", "")):
            n_jornalistas += 1
            continue
        uf = resolve_uf(env, speaker_map)
        if uf:
            ufs_encontradas.add(uf)
        mantidos.append(env)

    novo[node_key][sub_key] = mantidos
    return novo, ufs_encontradas, n_jornalistas


def separar_por_estado(dataset: list, envolvidos_path: tuple, usa_transcricao: bool, output_dir: Path):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    por_estado = defaultdict(list)
    sem_estado = []
    total_jornalistas = 0

    for record in dataset:
        novo, ufs, n_jorn = process_record(record, envolvidos_path, usa_transcricao)
        total_jornalistas += n_jorn
        if ufs:
            for uf in ufs:
                por_estado[uf].append(novo)
        else:
            sem_estado.append(novo)

    for uf, registros in sorted(por_estado.items()):
        with open(output_dir / f"{uf}.jsonl", "w", encoding="utf-8") as f:
            for r in registros:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    if sem_estado:
        with open(output_dir / "SEM_ESTADO.jsonl", "w", encoding="utf-8") as f:
            for r in sem_estado:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")

    resumo = {uf: len(regs) for uf, regs in sorted(por_estado.items())}
    resumo["SEM_ESTADO"] = len(sem_estado)
    return resumo, total_jornalistas


In [30]:
resumo_lds, jornalistas_removidos_lds = separar_por_estado(
    phbr_lds,
    envolvidos_path=("metadados", "envolvidos"),
    usa_transcricao=True,
    output_dir=OUTPUT_DIR / "PublicHearingBR_LDS_por_estado",
)

resumo_nli, jornalistas_removidos_nli = separar_por_estado(
    phbr_nli,
    envolvidos_path=("metadados_extraidos", "envolvidos"),
    usa_transcricao=False,
    output_dir=OUTPUT_DIR / "PublicHearingBR_NLI_por_estado",
)


## 6. Resumo final

In [31]:
def print_resumo(nome_dataset, resumo, jornalistas_removidos, total_original):
    print(f"=== {nome_dataset} ===")
    print(f"Registros originais: {total_original}")
    print(f"Envolvidos removidos por serem jornalistas: {jornalistas_removidos}")
    print(f"Registros sem nenhum estado identificado: {resumo.get('SEM_ESTADO', 0)}")
    print("Registros por estado (um registro pode contar em mais de um estado):")
    for uf, n in sorted(resumo.items()):
        if uf != "SEM_ESTADO":
            print(f"  {uf}: {n}")
    print()


print_resumo("PublicHearingBR_LDS", resumo_lds, jornalistas_removidos_lds, len(phbr_lds))
print_resumo("PublicHearingBR_NLI", resumo_nli, jornalistas_removidos_nli, len(phbr_nli))

print(f"Arquivos gerados em: {(OUTPUT_DIR / 'PublicHearingBR_LDS_por_estado').resolve()}")
print(f"Arquivos gerados em: {(OUTPUT_DIR / 'PublicHearingBR_NLI_por_estado').resolve()}")


=== PublicHearingBR_LDS ===
Registros originais: 206
Envolvidos removidos por serem jornalistas: 8
Registros sem nenhum estado identificado: 3
Registros por estado (um registro pode contar em mais de um estado):
  AC: 3
  AL: 5
  AM: 7
  AP: 2
  BA: 27
  CE: 19
  DF: 32
  ES: 12
  GO: 15
  MA: 4
  MG: 37
  MS: 5
  MT: 5
  PA: 8
  PB: 6
  PE: 16
  PI: 7
  PR: 16
  RJ: 47
  RN: 6
  RO: 1
  RR: 4
  RS: 24
  SC: 16
  SE: 3
  SP: 53
  TO: 3

=== PublicHearingBR_NLI ===
Registros originais: 206
Envolvidos removidos por serem jornalistas: 8
Registros sem nenhum estado identificado: 45
Registros por estado (um registro pode contar em mais de um estado):
  AC: 3
  AL: 4
  AM: 4
  AP: 4
  BA: 19
  CE: 12
  DF: 39
  ES: 6
  GO: 13
  MA: 3
  MG: 30
  MS: 3
  MT: 4
  PA: 6
  PB: 7
  PE: 13
  PI: 5
  PR: 12
  RJ: 36
  RN: 7
  RO: 2
  RR: 3
  RS: 19
  SC: 13
  SE: 3
  SP: 49

Arquivos gerados em: /home/lmmf/Documentos/kunumi/ideias_em_rede/OpenParla/Leo/output/PublicHearingBR_LDS_por_estado
Arquivos 